In [ ]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from torch.nn import Module, Sequential, Linear, LSTM
from torch.nn.utils import clip_grad_norm_
from torch.nn.functional import mse_loss
from sklearn.preprocessing import MinMaxScaler
from torch import Tensor
from typing import Iterator
Loader = Iterator[Tensor]

In [ ]:
PATH  = "NN-data-5-15-1.csv"
GROUP = ["GL"]
VAL   = "volume"
YEAR  = "year"
MONTH = "month"

In [ ]:
YSIZE  = 12
PYEAR  = 2025
EPOCHS = 220
HIDNS  = 200
BSIZE  = 4
LRATE  = 5e-3
SLEN   = 4
THRESH = 1e+5

<br>

### Organizing the data

In [ ]:
data = pd.read_csv(PATH, usecols = [ * GROUP, YEAR, MONTH, VAL])

In [ ]:
data = data[data[YEAR] <= PYEAR]

In [ ]:
data = data.groupby(GROUP).filter(lambda g : g.groupby(YEAR)[VAL].sum().mean() > THRESH)

Setting groups as features

In [ ]:
data = data.pivot_table(index = [YEAR, MONTH], columns = GROUP, values = VAL, aggfunc = 'sum', fill_value = 0).sort_index()

In [ ]:
train = data.loc[data.index.get_level_values(level = YEAR) != PYEAR]
test  = data.loc[data.index.get_level_values(level = YEAR) >= PYEAR - 1]

Filtering for full years only

In [ ]:
train = train.groupby(level = YEAR).filter(lambda g : g.count().min() == YSIZE)

Min-Max normalizing each group. Preserves order and sets a [0, 1] range.

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train);

Setting years as batches and months as the sequance.

In [ ]:
def pipeline(table: pd.DataFrame) -> Tensor:
    table = table.groupby(level = YEAR)
    table = table.apply(scaler.transform)
    table = table.apply(torch.FloatTensor)
    table = table.values.tolist()
    table = torch.stack([torch.cat([prev, curr]) for prev, curr in zip(table, table[1:])])
    return table

In [ ]:
train_ten = pipeline(train)
test_ten  = pipeline(test)

In [ ]:
target_ten = train_ten[:, 1:, :]
train_ten  = train_ten[:, :-1, :]

In [ ]:
assert train_ten.size(2) == target_ten.size(2) == test_ten.size(2)
assert train_ten.size(1) == target_ten.size(1) == 2 * YSIZE - 1
assert train_ten.size(0) == target_ten.size(0)
assert test_ten.size(0) == 1
assert SLEN < test_ten.size(1) - YSIZE

In [ ]:
tdataset = TensorDataset(train_ten, target_ten)
loader : Loader = DataLoader(tdataset, batch_size = BSIZE, shuffle = True)

<br>

### Building architactures

In [ ]:
class Forecaster(Module):
    def fit(self, loader: Loader, epochs: int, lr: float): pass
    def forecast(self, seed: Tensor, fh: int): pass

In [ ]:
class YearlyRNN(Forecaster):

    def __init__(self, input_dim: int, lstm_hid: int = 300):
        super().__init__()
        
        self.rnn = LSTM(input_dim, lstm_hid, batch_first = True)
        self.head = Sequential(Linear(lstm_hid, input_dim))


    def forward(self, x: Tensor, hid = None) -> tuple[Tensor, Tensor]:
        x, hid = self.rnn(x, hid)
        x = self.head(x)
        return x, hid


    def fit(self, loader: Loader, epochs: int, lr: float):
        
        optimizer = Adam(self.parameters(), lr=lr)
        self.train()


        def train_step(data_batch: Tensor, target_batch: Tensor) -> float:
            
            data_batch += torch.randn_like(data_batch) * 1e-2
            optimizer.zero_grad()
            preds, _ = self.forward(data_batch)
            loss = mse_loss(preds, target_batch)
            loss.backward()
            clip_grad_norm_(self.parameters(), max_norm = 1.0)
            optimizer.step()
            return loss.item()


        last_avg_loss = float('inf')
        strike_count = 0
        for epoch in range(1, epochs + 1):
            
            total_loss = sum(train_step(data_batch, target_batch) for data_batch, target_batch in loader)

            if epoch % 10 == 0:

                avg_loss = total_loss / len(loader)
                print(f"Epoch {epoch:3d} | Avg Loss: {avg_loss:.4f}")

                if avg_loss > last_avg_loss:
                    strike_count += 1
                    if strike_count >= 6:
                        print("Early stopping due to loss increase.")
                        break

                last_avg_loss = avg_loss


    @torch.no_grad
    def forecast(self, seed: Tensor, fh: int, hid = None):
        
        self.eval()

        for step in range(seed.size(1)):
            val = seed[:, step, :]
            yield val
            _, hid = self.forward(val, hid)

        for step in range(fh):
            val, hid = self.forward(val, hid)
            yield val

In [ ]:
model = YearlyRNN(input_dim = train_ten.size(-1), lstm_hid = HIDNS)

<br>

### Training

In [ ]:
model.fit(loader, EPOCHS, LRATE)

<br>

### Forecasting

In [ ]:
pred = torch.concat(tuple(model.forecast(test_ten[:, :YSIZE + SLEN, :], YSIZE - SLEN))).relu()

In [ ]:
test_ten = test_ten.squeeze()

In [ ]:
assert pred.size(0) == YSIZE * 2
assert pred.size(1) == test_ten.size(1)

In [ ]:
pred = pred.detach().numpy()
test_ten = test_ten.detach().numpy()

In [ ]:
pred = scaler.inverse_transform(pred)
test_ten = scaler.inverse_transform(test_ten)

In [ ]:
pred = pred[YSIZE:]
test_ten = test_ten[YSIZE:]
test = test.iloc[YSIZE:]

In [ ]:
def metrics(pred: Tensor, actual: Tensor):

    print(f"total prediction: {pred.sum():.2e}")
    print(f"total actual: {actual.sum():.2e}")

    err = abs(pred.sum() - actual.sum())

    print(f"total error: {err:.2e}")
    print(f"error percentage: {err / actual.sum() * 100:.2f}%")

In [ ]:
metrics(pred, test_ten)

In [ ]:
for group, x, y in zip(test.columns, pred.T, test_ten.T):

    if y.sum() == 0: continue

    print(group)
    metrics(x, y)